# Experiment 2 — Meteor-radar winds near Campina Grande

**Question:** How do meteor-echo counts, geometry, and number of radar sites affect mesospheric wind retrieval?

This self-contained OSSE models hypothetical independent monostatic radars near Campina Grande and other northeastern Brazilian cities. These are candidate sites, not claims of operating radars. Individual meteor trails provide radial drift velocities; weighted least squares retrieves horizontal wind components in altitude bins.

## Forward model and assumptions

For azimuth A clockwise from north and elevation e, \(v_r=\cos(e)[u\sin A+v\cos A]+\epsilon\), where all velocities/noise are in m s⁻¹. Each echo contributes a row \([\cos(e)\sin A,\cos(e)\cos A]\). Covariance is \((H^T R^{-1}H)^{-1}\). Vertical wind is assumed negligible. Echoes are independent; multiple sites sample distinct local volumes. A pooled regional estimate assumes spatially uniform wind, so spatial variation creates representativeness error rather than extra local information.

Echo azimuth/elevation distributions, meteor altitude, horizontal footprint, radial precision, and site coordinates are configurable assumptions. This OSSE does not model real antenna patterns, meteor chemistry, or hardware-specific selection.

In [1]:
from pathlib import Path
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED=20261108; rng=np.random.default_rng(SEED)
ROOT=Path(os.environ.get("SBGEA_KEYNOTE_DATA",Path.home()/"Documents/Science/Programs/LFS/KEYNOTE")).expanduser()
OUT=ROOT/"outputs"; FIG=OUT/"figures"/"keynote"; DATA=OUT/"data"/"osse"; FIG.mkdir(parents=True,exist_ok=True); DATA.mkdir(parents=True,exist_ok=True)
# Coordinates are approximate city centers; the added sites are hypothetical deployments.
SITE_LL={"Campina Grande":(-7.23,-35.88),"Patos":(-7.02,-37.28),"João Pessoa":(-7.12,-34.86),"Natal":(-5.79,-35.21)}
lat0,lon0=SITE_LL["Campina Grande"]
def ll_to_xy(lat,lon): return (lon-lon0)*111.0*np.cos(np.deg2rad(lat0)),(lat-lat0)*111.0
def xy_to_ll(x,y): return lat0+y/111.,lon0+x/(111*np.cos(np.deg2rad(lat0)))
SITE_XY={k:np.array(ll_to_xy(*v)) for k,v in SITE_LL.items()}
SIGMA_RADIAL=5.0

def wind_field(z_km,x_km,y_km):
    # Smooth illustrative regional field; gradients are m/s per km and per km altitude.
    return 25+.35*(z_km-90)+.035*x_km+.015*y_km, -10-.20*(z_km-90)+.010*x_km-.030*y_km

def sample_echoes(n,site_xy,seed,*,uniform_wind=True,az_center=None,az_halfwidth=180.,sigma=SIGMA_RADIAL):
    r=np.random.default_rng(seed); z=np.clip(r.normal(90.0,4.5,n),80.0,100.0)
    if az_center is None: az=r.uniform(0,360,n)
    else: az=(az_center+r.uniform(-az_halfwidth,az_halfwidth,n))%360
    el=r.uniform(20.,70.,n); azr=np.deg2rad(az); elr=np.deg2rad(el)
    slant_horizontal=z/np.tan(elr); x=site_xy[0]+slant_horizontal*np.sin(azr); y=site_xy[1]+slant_horizontal*np.cos(azr)
    if uniform_wind: xw=np.zeros(n); yw=np.zeros(n)
    else: xw=x; yw=y
    u,v=wind_field(z,xw,yw); h=np.column_stack([np.cos(elr)*np.sin(azr),np.cos(elr)*np.cos(azr)])
    radial=h[:,0]*u+h[:,1]*v+r.normal(0,sigma,n)
    return {"z":z,"az":az,"el":el,"x":x,"y":y,"H":h,"radial":radial,"sigma":np.full(n,sigma)}

def wls(H,y,sigma):
    H=np.asarray(H); y=np.asarray(y); w=1/np.asarray(sigma)**2
    normal=H.T@(w[:,None]*H); cond=float(np.linalg.cond(normal))
    cov=np.linalg.pinv(normal); estimate=cov@(H.T@(w*y))
    return estimate,cov,cond

def altitude_binned(track,edges):
    rows=[]
    for lo,hi in zip(edges[:-1],edges[1:]):
        use=(track["z"]>=lo)&(track["z"]<hi)
        if use.sum()<3: continue
        est,cov,cond=wls(track["H"][use],track["radial"][use],track["sigma"][use])
        rows.append({"z_mid_km":(lo+hi)/2,"n_echo":int(use.sum()),"east_m_s":est[0],"north_m_s":est[1],"sigma_east_m_s":np.sqrt(cov[0,0]),"sigma_north_m_s":np.sqrt(cov[1,1]),"condition":cond})
    return pd.DataFrame(rows)

# Scenario A: one all-sky site, spatially uniform wind to isolate count and geometry effects.
counts=np.array([10,25,50,100,200,500]); nmc=50; echo_rows=[]
for n in counts:
    errors=[]; formals=[]; conds=[]
    for r in range(nmc):
        tr=sample_echoes(int(n),SITE_XY["Campina Grande"],SEED+100*r+int(n),uniform_wind=True)
        est,cov,cond=wls(tr["H"],tr["radial"],tr["sigma"]); truth=np.array(wind_field(90,0,0))
        errors.append(np.linalg.norm(est-truth)); formals.append(np.sqrt(np.trace(cov))); conds.append(cond)
    echo_rows.append({"echoes":n,"vector_rmse_m_s":np.sqrt(np.mean(np.square(errors))),"formal_sigma_norm_m_s":np.median(formals),"median_condition":np.median(conds)})
echo_sens=pd.DataFrame(echo_rows); echo_sens.to_csv(DATA/"experiment_2_echo_sensitivity_reconstructed.csv",index=False)
# Deliberately narrow azimuth coverage shows that more measurements cannot cure poor geometry.
good=sample_echoes(200,SITE_XY["Campina Grande"],SEED+999,uniform_wind=True)
poor=sample_echoes(200,SITE_XY["Campina Grande"],SEED+1000,uniform_wind=True,az_center=0,az_halfwidth=12)
_,cov_good,cond_good=wls(good["H"],good["radial"],good["sigma"]); _,cov_poor,cond_poor=wls(poor["H"],poor["radial"],poor["sigma"])
print("200 echoes: all-sky condition",round(cond_good,2),"narrow-sector condition",round(cond_poor,2))

# Scenario B: each hypothetical radar is an independent monostatic site and samples its own trails.
per_site=60; site_rows=[]; pooled_by_n={}; mapped_final=[]
for nsite in range(1,5):
    tracks=[]
    for si,name in enumerate(list(SITE_LL)[:nsite]):
        tr=sample_echoes(per_site,SITE_XY[name],SEED+2000+si,uniform_wind=False)
        tracks.append((name,tr))
    allH=np.vstack([t["H"] for _,t in tracks]); ally=np.concatenate([t["radial"] for _,t in tracks]); alls=np.concatenate([t["sigma"] for _,t in tracks])
    est,cov,cond=wls(allH,ally,alls); regional=np.mean([wind_field(90,*SITE_XY[nm]) for nm,_ in tracks],axis=0); local=np.array(wind_field(90,0,0))
    local_tr=tracks[0][1]; _,local_cov,_=wls(local_tr["H"],local_tr["radial"],local_tr["sigma"])
    row={"sites":nsite,"echoes":nsite*per_site,"pooled_sigma_norm_m_s":np.sqrt(np.trace(cov),),"pooled_condition":cond,"pooled_offset_from_CG_m_s":np.linalg.norm(est-local),"regional_truth_offset_from_CG_m_s":np.linalg.norm(regional-local),"CG_local_sigma_norm_m_s":np.sqrt(np.trace(local_cov))}
    site_rows.append(row); pooled_by_n[nsite]=(est,cov,regional,local)
    if nsite==4: mapped_final=tracks.copy()
site_sens=pd.DataFrame(site_rows); site_sens.to_csv(DATA/"experiment_2_site_sensitivity_reconstructed.csv",index=False)
mapped=[(name,t["x"],t["y"]) for name,t in mapped_final]
# Cadence/precision/bins are explicit design knobs, not hidden fixed assumptions.
config_rows=[]
for sigma in [2.,5.,10.]:
    tr=sample_echoes(120,SITE_XY["Campina Grande"],SEED+int(sigma*10),uniform_wind=True,sigma=sigma); _,cv,co=wls(tr["H"],tr["radial"],tr["sigma"])
    config_rows.append({"usable_echoes":120,"radial_sigma_m_s":sigma,"bin_width_km":20.,"formal_sigma_norm_m_s":np.sqrt(np.trace(cv)),"condition":co})
for width in [5.,10.,20.]:
    tr=sample_echoes(500,SITE_XY["Campina Grande"],SEED+int(width*100),uniform_wind=True)
    binned=altitude_binned(tr,np.arange(80,100+width,width)); config_rows.append({"usable_echoes":int(binned.n_echo.sum()),"radial_sigma_m_s":SIGMA_RADIAL,"bin_width_km":width,"formal_sigma_norm_m_s":float(np.sqrt(np.mean(binned.sigma_east_m_s**2+binned.sigma_north_m_s**2))),"condition":float(binned.condition.median())})
pd.DataFrame(config_rows).to_csv(DATA/"experiment_2_design_sensitivity_reconstructed.csv",index=False)
# Reproducibility and geometry checks.
assert np.all(np.diff(echo_sens.formal_sigma_norm_m_s)<0)
assert cond_poor>cond_good*5
tr0=sample_echoes(30,SITE_XY["Campina Grande"],123,uniform_wind=True); tr1=sample_echoes(30,SITE_XY["Campina Grande"],123,uniform_wind=True); assert np.array_equal(tr0["radial"],tr1["radial"])
np.savez_compressed(DATA/"experiment_2_payload_reconstructed.npz",echo_counts=counts,rmse=echo_sens.vector_rmse_m_s,formal_sigma=echo_sens.formal_sigma_norm_m_s,site_count=site_sens.sites,pooled_sigma=site_sens.pooled_sigma_norm_m_s,pooled_offset=site_sens.pooled_offset_from_CG_m_s,site_xy=np.array([SITE_XY[k] for k in SITE_LL]),meteor_xy=np.concatenate([np.column_stack([x,y]) for _,x,y in mapped]).ravel(),meteor_z=np.concatenate([t["z"] for _,t in tracks]),altitude_km=np.linspace(82.5,97.5,4),east_true=np.array([wind_field(z,0,0)[0] for z in [82.5,87.5,92.5,97.5]]),east_fit=np.array([altitude_binned(sample_echoes(500,SITE_XY["Campina Grande"],SEED+z,uniform_wind=False),np.arange(80,101,5)).east_m_s.iloc[i] for i,z in enumerate(range(4))]))
print("Scenario A rows:",len(echo_sens),"Scenario B rows:",len(site_sens),"radar sites are independent monostatic instruments")

200 echoes: all-sky condition 1.07 narrow-sector condition 67.98
Scenario A rows: 6 Scenario B rows: 4 radar sites are independent monostatic instruments


In [2]:
# Slide figure: candidate network/map and actual Monte Carlo sensitivity results.
fig=plt.figure(figsize=(11.25,5.5),dpi=300,facecolor="white"); gs=fig.add_gridspec(1,2,left=.06,right=.97,top=.82,bottom=.22,wspace=.27,width_ratios=[1,1.35]); axm=fig.add_subplot(gs[0,0]); axr=fig.add_subplot(gs[0,1])
fig.suptitle("RADAR METEÓRICO · Campina Grande e sítios hipotéticos",x=.06,ha="left",fontsize=17,fontweight="bold",color="#082F73")
fig.text(.06,.875,"Quantas observações precisamos para medir o vento?",fontsize=11,color="#25364A")
allx=np.concatenate([x for _,x,y in mapped]); ally=np.concatenate([y for _,x,y in mapped]); mlat,mlon=xy_to_ll(allx,ally); axm.scatter(mlon,mlat,s=5,color="#9DB4C5",alpha=.38,label="meteoros simulados")
label_offsets={"Campina Grande":(5,-20),"Patos":(-34,7),"João Pessoa":(6,6),"Natal":(6,7)}
for name,(lat,lon) in SITE_LL.items():
    col="#082F73" if name=="Campina Grande" else "#EC8135"
    axm.scatter(lon,lat,s=50 if name=="Campina Grande" else 34,color=col,edgecolor="white",zorder=4)
    axm.annotate(name,(lon,lat),xytext=label_offsets[name],textcoords="offset points",fontsize=6.5,color=col)
axm.set(xlabel="Longitude (°)",ylabel="Latitude (°)",xlim=(-41,-32),ylim=(-12,-2)); axm.grid(alpha=.18); axm.set_title("Campina Grande + candidatos",loc="left",fontsize=10,color="#082F73")
axr.errorbar(counts,echo_sens.vector_rmse_m_s,yerr=echo_sens.formal_sigma_norm_m_s,fmt="o-",color="#2686D0",capsize=3,label="um radar · RMSE (barras: σ formal)")
ax2=axr.twinx(); ax2.plot(site_sens.sites,site_sens.pooled_sigma_norm_m_s,"s--",color="#70AE54",label="rede · σ formal agrupado")
axr.set_xscale("log"); axr.set(xlabel="Ecos utilizáveis por intervalo (um radar)",ylabel="Erro vetorial / incerteza (m/s)"); ax2.set(xlabel="",ylabel="σ formal agrupado (m/s)"); axr.grid(alpha=.2)
axr.set_title("Contagem e número de sítios",loc="left",fontsize=10,color="#082F73")
lines,labels=axr.get_legend_handles_labels(); l2,lab2=ax2.get_legend_handles_labels(); axr.legend(lines+l2,labels+lab2,frameon=False,fontsize=8,loc="upper right")
fig.text(.06,.12,f"500 ecos: RMSE {echo_sens.vector_rmse_m_s.iloc[-1]:.1f} m/s. Mais sítios melhoram a precisão regional, mas o vento local de Campina Grande não ganha ecos.",fontsize=8.3,color="#25364A")
fig.text(.06,.06,"Cada sítio adicional é um radar monostático independente; campos regionais variáveis criam viés ao agrupar.",fontsize=7.5,color="#63758A")
fig.savefig(FIG/"Experiment_2_Slide.png",facecolor="white"); fig.savefig(FIG/"Experiment_2_Slide_Recovered.png",facecolor="white"); plt.show()
fig2,ax=plt.subplots(figsize=(7,4)); ax.plot(counts,echo_sens.formal_sigma_norm_m_s,"o-",label="σ formal"); ax.plot(counts,echo_sens.vector_rmse_m_s,"s--",label="RMSE Monte Carlo"); ax.set_xscale("log"); ax.set(xlabel="Ecos por intervalo",ylabel="Erro vetorial (m/s)",title="Mais ecos reduzem ruído quando a geometria é adequada"); ax.grid(alpha=.2); ax.legend(frameon=False); fig2.tight_layout(); fig2.savefig(FIG/"Experiment_2_Sensitivity_Recovered.png",dpi=180); plt.show()
print("Saved reconstructed Experiment 2 tables, payload, and slide.")

Saved reconstructed Experiment 2 tables, payload, and slide.


/var/folders/6m/4lrc3ql56_g0hsgdbnj5t18r0000gn/T/ipykernel_80546/3191917742.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(FIG/"Experiment_2_Slide.png",facecolor="white"); fig.savefig(FIG/"Experiment_2_Slide_Recovered.png",facecolor="white"); plt.show()
/var/folders/6m/4lrc3ql56_g0hsgdbnj5t18r0000gn/T/ipykernel_80546/3191917742.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig2,ax=plt.subplots(figsize=(7,4)); ax.plot(counts,echo_sens.formal_sigma_norm_m_s,"o-",label="σ formal"); ax.plot(counts,echo_sens.vector_rmse_m_s,"s--",label="RMSE Monte Carlo"); ax.set_xscale("log"); ax.set(xlabel="Ecos por intervalo",ylabel="Erro vetorial (m/s)",title="Mais ecos reduzem ruído quando a geometria é adequada"); ax.grid(alpha=.2); ax.legend(frameon=False); fig2.tight_layout(); fig2.savefig(FIG/"Experiment_2_Sensitivity_Recovered.png",dpi=180); plt.show()


## Interpretation, limitations, and references

More echoes reduce measurement noise when azimuth coverage constrains both components. Narrow azimuth sectors remain ill-conditioned even with many measurements. In the multi-site scenario, the instruments are independent monostatic radars with distinct meteor samples; extra sites improve regional pooled precision but can make that estimate differ from the local Campina Grande wind. Duration is represented by the usable echo count; real echo selection depends on observing geometry, meteor rates, and hardware.

References: [Atmospheric Measurement Techniques (2022)](https://amt.copernicus.org/articles/15/5769/2022/), [Journal of Atmospheric and Solar-Terrestrial Physics](https://doi.org/10.1016/S1364-6826(00)00138-3).

### What could an undergraduate investigate next?
1. How does a narrow azimuth sector affect the covariance of the two wind components?
2. How should altitude-bin width balance vertical resolution against echo count?
3. When is a regional multi-radar average a poor estimate of the local wind?

### Recovery provenance

This executed notebook is a reconstructed, reproducible implementation informed by the project specification and preserved OSSE tables/payloads. The original notebook bytes were deleted during migration. Their original SHA-256 values remain in `KEYNOTE/MIGRATION_MANIFEST.csv`; the rebuilt source is not represented as a byte-identical recovery.

## Interactive radar experiment design

Change the number of usable echoes, radial-velocity precision, and azimuth coverage. The weighted least-squares estimate and its covariance are recomputed using the notebook's meteor forward model. The narrow-sector setting demonstrates why many observations cannot compensate for poor viewing geometry.

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
radar_out=widgets.Output()
def _radar_explore(n_echoes=120, radial_sigma=5.0, azimuth_halfwidth=180):
    with radar_out:
        clear_output(wait=True)
        track=sample_echoes(int(n_echoes),SITE_XY["Campina Grande"],SEED+808,
            uniform_wind=True,az_center=0.0,az_halfwidth=float(azimuth_halfwidth),sigma=float(radial_sigma))
        estimate,cov,condition=wls(track["H"],track["radial"],track["sigma"])
        truth=np.array(wind_field(90,0,0))
        fig,ax=plt.subplots(1,2,figsize=(8,3.0),facecolor="white")
        ax[0].scatter(track["az"],track["el"],s=12,c=track["radial"],cmap="coolwarm")
        ax[0].set(xlabel="Azimute (°)",ylabel="Elevação (°)",xlim=(0,360),ylim=(0,90)); ax[0].set_title("Geometria dos ecos"); ax[0].grid(alpha=.2)
        ax[1].errorbar(["Leste","Norte"],estimate,yerr=np.sqrt(np.diag(cov)),fmt="o",capsize=4,color="#2686D0",label="recuperado")
        ax[1].scatter(["Leste","Norte"],truth,color="#EC8135",marker="x",s=50,label="vento nominal")
        ax[1].set_ylabel("Vento (m/s)"); ax[1].set_title(f"condição geométrica = {condition:.1f}"); ax[1].grid(axis="y",alpha=.2); ax[1].legend(frameon=False)
        fig.tight_layout(); plt.show()
        print("Estimativa E/N:",np.round(estimate,2),"m/s | σ:",np.round(np.sqrt(np.diag(cov)),2),"m/s")
controls=widgets.interactive(_radar_explore,
    n_echoes=widgets.IntSlider(value=120,min=10,max=500,step=10,description="Ecos"),
    radial_sigma=widgets.FloatSlider(value=5.0,min=1.0,max=15.0,step=.5,description="σ radial"),
    azimuth_halfwidth=widgets.IntSlider(value=180,min=10,max=180,step=10,description="Setor ±°"))
display(controls)


interactive(children=(IntSlider(value=120, description='Ecos', max=500, min=10, step=10), FloatSlider(value=5.…